# ENV ET OUTILS

In [74]:
import torch # pour la manipulation des tenseurs
import torch.nn as nn # pour la création de modèles de réseaux de neurones
import torch.optim as optim # pour l'optimisation des paramètres du modèle

# 1. Nos données d'entraînement (Texte, Tags)
# On utilise un exemple simple en français pour commencer
training_data = [
    ("Amadou étudie le bambara".split(), ["NOM", "VERBE", "DET", "NOM"]),
    ("le modèle comprend les phrases".split(), ["DET", "NOM", "VERBE", "DET", "NOM"])
]

# 2. Création du dictionnaire de mots (Word to Index)
word_to_ix = {}
for sentence, tags in training_data:
    for word in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

# On ajoute un token spécial pour les mots inconnus (Out Of Vocabulary)
word_to_ix["<UNK>"] = len(word_to_ix)

# 3. Création du dictionnaire d'étiquettes (Tag to Index)
tag_to_ix = {"NOM": 0, "VERBE": 1, "DET": 2}

print("Dictionnaire des mots (word_to_ix) :")
print(word_to_ix)
print("\nDictionnaire des étiquettes (tag_to_ix) :")
print(tag_to_ix)

"""
en gros le dictionnaire des mots (word_to_ix) est utilisé pour convertir les mots en indices numériques, ce qui est nécessaire pour l'entrée dans le modèle de réseau de neurones. 
Chaque mot unique dans les données d'entraînement se voit attribuer un index unique. 
Le token spécial "<UNK>" est utilisé pour représenter les mots qui ne sont pas présents dans le dictionnaire, ce qui permet au modèle de gérer les mots inconnus lors de l'inférence.    
"""

Dictionnaire des mots (word_to_ix) :
{'Amadou': 0, 'étudie': 1, 'le': 2, 'bambara': 3, 'modèle': 4, 'comprend': 5, 'les': 6, 'phrases': 7, '<UNK>': 8}

Dictionnaire des étiquettes (tag_to_ix) :
{'NOM': 0, 'VERBE': 1, 'DET': 2}


'\nen gros le dictionnaire des mots (word_to_ix) est utilisé pour convertir les mots en indices numériques, ce qui est nécessaire pour l\'entrée dans le modèle de réseau de neurones. \nChaque mot unique dans les données d\'entraînement se voit attribuer un index unique. \nLe token spécial "<UNK>" est utilisé pour représenter les mots qui ne sont pas présents dans le dictionnaire, ce qui permet au modèle de gérer les mots inconnus lors de l\'inférence.    \n'

In [75]:
# Fonction pour convertir une liste de mots en tenseur d'index
def prepare_sequence(seq, to_ix):
    idxs = []
    for w in seq:
        if w in to_ix:
            idxs.append(to_ix[w])
        else:
            idxs.append(to_ix["<UNK>"])  # Si le mot est inconnu
    return torch.tensor(idxs, dtype=torch.long)

# Test de la fonction
exemple_phrase = "Amadou comprend le bambara".split()
tensor_phrase = prepare_sequence(exemple_phrase, word_to_ix)

print("Phrase d'origine :", exemple_phrase)
print("Tenseur PyTorch généré :", tensor_phrase)

"""
ici , la fonction prepare_sequence prend une séquence de mots et le dictionnaire word_to_ix comme entrées.
Elle convertit chaque mot en son index correspondant dans le dictionnaire.
Si un mot n'est pas trouvé dans le dictionnaire, il est remplacé par l'index du token spécial "<UNK>".
Le résultat est un tenseur PyTorch contenant les indices des mots, prêt à être utilisé comme entrée pour un modèle de réseau de neurones.

"""

Phrase d'origine : ['Amadou', 'comprend', 'le', 'bambara']
Tenseur PyTorch généré : tensor([0, 5, 2, 3])


'\nici , la fonction prepare_sequence prend une séquence de mots et le dictionnaire word_to_ix comme entrées.\nElle convertit chaque mot en son index correspondant dans le dictionnaire.\nSi un mot n\'est pas trouvé dans le dictionnaire, il est remplacé par l\'index du token spécial "<UNK>".\nLe résultat est un tenseur PyTorch contenant les indices des mots, prêt à être utilisé comme entrée pour un modèle de réseau de neurones.\n\n'

In [76]:
# Définition du modèle LSTM pour le POS tagging
class LSTMTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size): 
        super(LSTMTagger, self).__init__() # 
        self.hidden_dim = hidden_dim

        # 1. La couche d'Embedding : convertit les index des mots en vecteurs denses
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # 2. La couche LSTM : prend les embeddings en entrée et renvoie les états cachés
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        # 3. La couche Linéaire : projette l'espace des états cachés vers l'espace des tags
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        # Passage avant (forward pass) :
        # On transforme la phrase d'index en embeddings
        embeds = self.word_embeddings(sentence)
        
        # Le LSTM attend un tenseur à 3 dimensions : (longueur_sequence, taille_batch, taille_embedding)
        # On utilise view() pour ajouter la dimension de batch (qui est de 1 ici)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        
        # On passe les sorties du LSTM à la couche linéaire pour obtenir les scores de chaque tag
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        
        # On applique un LogSoftmax pour obtenir des log-probabilités
        tag_scores = torch.log_softmax(tag_space, dim=1)
        return tag_scores
    
"""
en gros , ce modèle LSTMTagger prend en entrée une phrase représentée par des indices de mots, la passe à travers une couche d'embedding, puis à travers un LSTM, 
et enfin projette les états cachés du LSTM vers l'espace des tags pour obtenir les scores de chaque tag.

ensuite , on applique une fonction LogSoftmax pour obtenir des log-probabilités pour chaque tag, ce qui est utile pour la classification multi-classes.

"""

"\nen gros , ce modèle LSTMTagger prend en entrée une phrase représentée par des indices de mots, la passe à travers une couche d'embedding, puis à travers un LSTM, \net enfin projette les états cachés du LSTM vers l'espace des tags pour obtenir les scores de chaque tag.\n\nensuite , on applique une fonction LogSoftmax pour obtenir des log-probabilités pour chaque tag, ce qui est utile pour la classification multi-classes.\n\n"

In [77]:
# 1. Définition des hyperparamètres (les dimensions de notre réseau)
EMBEDDING_DIM = 6  # Chaque mot sera représenté par un vecteur de taille 6
HIDDEN_DIM = 6     # Taille de la mémoire interne (état caché) du LSTM

# 2. Initialisation du modèle, de la fonction de perte (Loss) et de l'optimiseur
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss() # Negative Log Likelihood Loss (idéal avec LogSoftmax)
optimizer = optim.SGD(model.parameters(), lr=0.1) # Descente de gradient stochastique

# 3. Test de prédiction AVANT entraînement
with torch.no_grad(): # On désactive le calcul des gradients (on ne fait pas d'entraînement ici)
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)
    
    print("Phrase testée :", training_data[0][0])
    print("\nScores bruts pour chaque tag (log-probabilités) :")
    print(tag_scores)
    
    # Pour chaque mot, on récupère l'index du tag qui a la plus grande probabilité
    _, predicted_tags = torch.max(tag_scores, dim=1)
    print("\nTags prédits (aléatoires pour l'instant) :", predicted_tags.tolist())
    
"""
ici , on teste le modèle avant l'entraînement pour voir les scores de log-probabilités pour chaque tag.
On utilise torch.no_grad() pour désactiver le calcul des gradients, car nous ne faisons pas d'entraînement à ce stade.
On prépare la séquence d'entrée, on passe cette séquence à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag, et enfin, on récupère les indices des tags prédits en prenant l'argmax de ces scores    
"""

Phrase testée : ['Amadou', 'étudie', 'le', 'bambara']

Scores bruts pour chaque tag (log-probabilités) :
tensor([[-1.1210, -1.2115, -0.9774],
        [-0.9839, -1.2450, -1.0841],
        [-0.9866, -1.2127, -1.1093],
        [-1.0260, -1.2510, -1.0347]])

Tags prédits (aléatoires pour l'instant) : [2, 0, 0, 0]


"\nici , on teste le modèle avant l'entraînement pour voir les scores de log-probabilités pour chaque tag.\nOn utilise torch.no_grad() pour désactiver le calcul des gradients, car nous ne faisons pas d'entraînement à ce stade.\nOn prépare la séquence d'entrée, on passe cette séquence à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag, et enfin, on récupère les indices des tags prédits en prenant l'argmax de ces scores    \n"

In [78]:
# On va entraîner le modèle sur 300 époques (passages complets sur le dataset)
print("Début de l'entraînement...")

for epoch in range(300):
    for sentence, tags in training_data:
        # Étape 1 : PyTorch accumule les gradients, on doit les remettre à zéro 
        # avant chaque phrase pour ne pas mélanger les calculs
        model.zero_grad()

        # Étape 2 : On prépare nos entrées (mots -> index) et nos cibles (tags -> index)
        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        # Étape 3 : Passage avant (Forward pass) pour obtenir les prédictions
        tag_scores = model(sentence_in)

        # Étape 4 : Calcul de la perte (l'erreur commise par le modèle)
        loss = loss_function(tag_scores, targets)

        # Étape 5 : Rétropropagation (Backward pass) pour calculer les gradients
        loss.backward()

        # Étape 6 : Mise à jour des poids du modèle
        optimizer.step()
        
    # On affiche l'évolution de la perte toutes les 50 époques
    if (epoch + 1) % 50 == 0:
        print(f"Époque {epoch + 1}/300 | Perte (Loss) : {loss.item():.4f}")

print("Entraînement terminé !")

"""
en gros , ici , on entraîne le modèle sur 300 époques. Pour chaque phrase dans les données d'entraînement, on effectue les étapes suivantes :
1. On remet à zéro les gradients accumulés.
2. On prépare les entrées et les cibles en convertissant les mots et les tags en indices.
3. On passe la phrase à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag.
4. On calcule la perte entre les prédictions du modèle et les tags réels.
5. On effectue la rétropropagation pour calculer les gradients.
6. On met à jour les poids du modèle en utilisant l'optimiseur.
On affiche la perte toutes les 50 époques pour suivre l'évolution de l'entraînement
    
"""

Début de l'entraînement...
Époque 50/300 | Perte (Loss) : 0.9474
Époque 100/300 | Perte (Loss) : 0.5166
Époque 150/300 | Perte (Loss) : 0.1364
Époque 200/300 | Perte (Loss) : 0.0582
Époque 250/300 | Perte (Loss) : 0.0355
Époque 300/300 | Perte (Loss) : 0.0252
Entraînement terminé !


"\nen gros , ici , on entraîne le modèle sur 300 époques. Pour chaque phrase dans les données d'entraînement, on effectue les étapes suivantes :\n1. On remet à zéro les gradients accumulés.\n2. On prépare les entrées et les cibles en convertissant les mots et les tags en indices.\n3. On passe la phrase à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag.\n4. On calcule la perte entre les prédictions du modèle et les tags réels.\n5. On effectue la rétropropagation pour calculer les gradients.\n6. On met à jour les poids du modèle en utilisant l'optimiseur.\nOn affiche la perte toutes les 50 époques pour suivre l'évolution de l'entraînement\n\n"

In [79]:
# On repasse le modèle en mode évaluation
model.eval()

# 1. Test sur une phrase d'entraînement
phrase_connue = "Amadou étudie le bambara".split()
with torch.no_grad():
    inputs = prepare_sequence(phrase_connue, word_to_ix)
    scores = model(inputs)
    _, predictions = torch.max(scores, dim=1)
    
    # On reconstruit les noms des tags pour que ce soit lisible
    # On inverse le dictionnaire tag_to_ix pour avoir {index: "NOM"}
    ix_to_tag = {v: k for k, v in tag_to_ix.items()}
    tags_predits = [ix_to_tag[idx.item()] for idx in predictions]
    
    print("--- TEST 1 (Phrase connue) ---")
    print("Phrase :", phrase_connue)
    print("Tags prédits :", tags_predits)

print("\n" + "="*40 + "\n")

# 2. Test avec un mot totalement inconnu ("ordinateur")
# Notre dictionnaire ne connaît pas "ordinateur", il va donc le remplacer par <UNK>
phrase_inconnue = "Amadou comprend l' ordinateur".split() # (On sépare l'apostrophe pour simplifier)
with torch.no_grad():
    inputs_inc = prepare_sequence(phrase_inconnue, word_to_ix)
    scores_inc = model(inputs_inc)
    _, predictions_inc = torch.max(scores_inc, dim=1)
    tags_predits_inc = [ix_to_tag[idx.item()] for idx in predictions_inc]
    
    print("--- TEST 2 (Avec mot inconnu) ---")
    print("Phrase :", phrase_inconnue)
    print("Tags prédits :", tags_predits_inc)

--- TEST 1 (Phrase connue) ---
Phrase : ['Amadou', 'étudie', 'le', 'bambara']
Tags prédits : ['NOM', 'VERBE', 'DET', 'NOM']


--- TEST 2 (Avec mot inconnu) ---
Phrase : ['Amadou', 'comprend', "l'", 'ordinateur']
Tags prédits : ['NOM', 'VERBE', 'NOM', 'NOM']


# PADDING ET BATCH

In [80]:
# 1. On ajoute de nouvelles phrases de longueurs très différentes
training_data_multisize = [
    ("Amadou étudie le bambara".split(), ["NOM", "VERBE", "DET", "NOM"]),
    ("le modèle comprend les phrases".split(), ["DET", "NOM", "VERBE", "DET", "NOM"]), # Phrase plus longue
    ("il apprend".split(), ["PRON", "VERBE"]) # Phrase très courte
]

# 2. On reconstruit notre dictionnaire de mots
word_to_ix_pad = {}
# IMPORTANT : On réserve l'index 0 pour le token de Padding <PAD>
word_to_ix_pad["<PAD>"] = 0
word_to_ix_pad["<UNK>"] = 1

for sentence, tags in training_data_multisize:
    for word in sentence:
        if word not in word_to_ix_pad:
            word_to_ix_pad[word] = len(word_to_ix_pad)

# 3. Dictionnaire de tags (on réserve aussi l'index 0 pour le padding des tags si besoin)
tag_to_ix_pad = {"<PAD>": 0, "NOM": 1, "VERBE": 2, "DET": 3, "PRON": 4}

print("Nouveau dictionnaire de mots (avec <PAD> à l'index 0) :")
print(word_to_ix_pad)

"""
en résumé , ici , on a ajouté de nouvelles phrases de longueurs différentes pour enrichir notre jeu de données d'entraînement.
On a reconstruit le dictionnaire de mots pour inclure un token spécial de padding "<PAD>" à l'index 0, ce qui est utile pour gérer les séquences de longueurs variables lors de l'entraînement des modèles de réseaux de neurones.
On a également ajouté un token spécial "<UNK>" pour les mots inconnus, et on a mis à jour le dictionnaire des tags pour inclure un index de padding si nécessaire. Cela permet de mieux gérer les séquences de différentes longueurs et d'améliorer la robustesse du modèle lors de l'entraînement et de l'inférence.

"""

Nouveau dictionnaire de mots (avec <PAD> à l'index 0) :
{'<PAD>': 0, '<UNK>': 1, 'Amadou': 2, 'étudie': 3, 'le': 4, 'bambara': 5, 'modèle': 6, 'comprend': 7, 'les': 8, 'phrases': 9, 'il': 10, 'apprend': 11}


'\nen résumé , ici , on a ajouté de nouvelles phrases de longueurs différentes pour enrichir notre jeu de données d\'entraînement.\nOn a reconstruit le dictionnaire de mots pour inclure un token spécial de padding "<PAD>" à l\'index 0, ce qui est utile pour gérer les séquences de longueurs variables lors de l\'entraînement des modèles de réseaux de neurones.\nOn a également ajouté un token spécial "<UNK>" pour les mots inconnus, et on a mis à jour le dictionnaire des tags pour inclure un index de padding si nécessaire. Cela permet de mieux gérer les séquences de différentes longueurs et d\'améliorer la robustesse du modèle lors de l\'entraînement et de l\'inférence.\n\n'

In [81]:
from torch.nn.utils.rnn import pad_sequence # pour le padding des séquences

# 1. Convertir toutes nos phrases et nos tags en listes de tenseurs d'index
sequences_mots = [] # pour stocker les séquences de mots converties en tenseurs
sequences_tags = [] # poir stocker les séquences de tags converties en tenseurs

for sentence, tags in training_data_multisize:
    # On utilise notre fonction de préparation avec le nouveau dictionnaire
    seq_tensor = prepare_sequence(sentence, word_to_ix_pad)
    tag_tensor = prepare_sequence(tags, tag_to_ix_pad)
    
    sequences_mots.append(seq_tensor)
    sequences_tags.append(tag_tensor)

# 2. Appliquer le Padding
# batch_first=True permet d'avoir une forme (taille_batch, longueur_sequence)
# padding_value=0 car 0 est l'index de <PAD>
padded_mots = pad_sequence(sequences_mots, batch_first=True, padding_value=0)
padded_tags = pad_sequence(sequences_tags, batch_first=True, padding_value=0)

# 3. Affichage du résultat
print("--- PHRASES RAMBOURRÉES (PADDED MOTS) ---")
print(padded_mots)
print("\nForme (Shape) du tenseur des mots :", padded_mots.shape)

print("\n" + "="*40 + "\n")

print("--- TAGS RAMBOURRÉS (PADDED TAGS) ---")
print(padded_tags)
print("\nForme (Shape) du tenseur des tags :", padded_tags.shape)

# les matrices viennent de la partie précédente, où nous avons converti les phrases et les tags en tenseurs d'index et appliqué un padding pour uniformiser la longueur des séquences.

"""
ici , on a converti toutes les phrases et leurs tags correspondants en tenseurs d'index, puis on a appliqué un padding pour que toutes les séquences aient la même longueur.
Le padding est fait avec l'index 0, qui correspond au token spécial "<PAD>".
Cela permet de créer des lots (batches) de données d'entrée et de sortie de taille uniforme, ce qui est nécessaire pour l'entraînement des modèles de réseaux de neurones.
"""

--- PHRASES RAMBOURRÉES (PADDED MOTS) ---
tensor([[ 2,  3,  4,  5,  0],
        [ 4,  6,  7,  8,  9],
        [10, 11,  0,  0,  0]])

Forme (Shape) du tenseur des mots : torch.Size([3, 5])


--- TAGS RAMBOURRÉS (PADDED TAGS) ---
tensor([[1, 2, 3, 1, 0],
        [3, 1, 2, 3, 1],
        [4, 2, 0, 0, 0]])

Forme (Shape) du tenseur des tags : torch.Size([3, 5])


'\nici , on a converti toutes les phrases et leurs tags correspondants en tenseurs d\'index, puis on a appliqué un padding pour que toutes les séquences aient la même longueur.\nLe padding est fait avec l\'index 0, qui correspond au token spécial "<PAD>".\nCela permet de créer des lots (batches) de données d\'entrée et de sortie de taille uniforme, ce qui est nécessaire pour l\'entraînement des modèles de réseaux de neurones.\n'

In [82]:
# 1. On redéfinit la fonction de perte pour ignorer le padding (index 0)
loss_function_pad = nn.NLLLoss(ignore_index=0)

# 2. Petite modification du modèle pour accepter les dimensions de batch
class LSTMTaggerWithBatch(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTaggerWithBatch, self).__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0) # padding_idx=0 met le vecteur du PAD à zéro
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True) # batch_first=True pour accepter la forme (batch, seq_len)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentences):
        # sentences a maintenant la forme: (batch_size, sequence_length)
        embeds = self.word_embeddings(sentences)
        
        # lstm_out aura la forme: (batch_size, sequence_length, hidden_dim)
        lstm_out, _ = self.lstm(embeds)
        
        # On passe toutes les sorties dans la couche linéaire
        tag_space = self.hidden2tag(lstm_out)
        
        # On applique le softmax sur la dernière dimension (les scores des tags)
        tag_scores = torch.log_softmax(tag_space, dim=-1)
        return tag_scores
    

"""
en gros , ce modèle LSTMTaggerWithBatch est une version modifiée du modèle précédent pour gérer les entrées par lots (batches).
La couche d'embedding est configurée pour ignorer le token de padding (index 0), et le LSTM est défini avec batch_first=True pour accepter des entrées de forme (batch_size, sequence_length, embedding_dim).
Le reste du modèle fonctionne de manière similaire, en projetant les sorties du LSTM vers l'espace des tags et en appliquant un LogSoftmax pour obtenir des log-probabilités pour chaque tag.

"""

"\nen gros , ce modèle LSTMTaggerWithBatch est une version modifiée du modèle précédent pour gérer les entrées par lots (batches).\nLa couche d'embedding est configurée pour ignorer le token de padding (index 0), et le LSTM est défini avec batch_first=True pour accepter des entrées de forme (batch_size, sequence_length, embedding_dim).\nLe reste du modèle fonctionne de manière similaire, en projetant les sorties du LSTM vers l'espace des tags et en appliquant un LogSoftmax pour obtenir des log-probabilités pour chaque tag.\n\n"

In [83]:
# 1. Hyperparamètres
EMBEDDING_DIM = 6
HIDDEN_DIM = 6

# 2. Initialisation du modèle avec batch et de l'optimiseur
model_batch = LSTMTaggerWithBatch(
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    vocab_size=len(word_to_ix_pad),
    tagset_size=len(tag_to_ix_pad)
)

optimizer_batch = optim.SGD(model_batch.parameters(), lr=0.1)

# 3. Passage de notre batch de mots dans le modèle
with torch.no_grad():
    predictions_batch = model_batch(padded_mots)

print("Forme de notre batch d'entrée :", padded_mots.shape)
print("Forme des scores de sortie    :", predictions_batch.shape)

"""
en gros , ce code montre comment passer un batch de phrases (avec padding) à travers le modèle LSTMTaggerWithBatch.
On utilise torch.no_grad() pour désactiver le calcul des gradients, car nous ne faisons pas d'entraînement à ce stade.
Le tenseur padded_mots contient les phrases avec padding, et predictions_batch contient les scores de log-probabilités pour chaque tag pour chaque mot dans le batch.
On affiche les formes (shapes) de ces tenseurs pour vérifier que tout est correct
La sortie doit avoir une forme de torch.Size([3, 5, 5]) (soit : [3 phrases, 5 mots par phrase, 5 étiquettes de tags possibles dans notre dictionnaire]).
    """

Forme de notre batch d'entrée : torch.Size([3, 5])
Forme des scores de sortie    : torch.Size([3, 5, 5])


"\nen gros , ce code montre comment passer un batch de phrases (avec padding) à travers le modèle LSTMTaggerWithBatch.\nOn utilise torch.no_grad() pour désactiver le calcul des gradients, car nous ne faisons pas d'entraînement à ce stade.\nLe tenseur padded_mots contient les phrases avec padding, et predictions_batch contient les scores de log-probabilités pour chaque tag pour chaque mot dans le batch.\nOn affiche les formes (shapes) de ces tenseurs pour vérifier que tout est correct\nLa sortie doit avoir une forme de torch.Size([3, 5, 5]) (soit : [3 phrases, 5 mots par phrase, 5 étiquettes de tags possibles dans notre dictionnaire]).\n    "

# PARTIE TOKENISATION

In [84]:
import re # pour les expressions régulières
import nltk # pour le tokenization et le traitement du langage naturel
from nltk.tokenize import WordPunctTokenizer # pour le tokenization basé sur les mots et la ponctuation
import spacy # pour le traitement du langage naturel avancé (tokenization, POS tagging, etc.............)

# On s'assure d'avoir les ressources NLTK nécessaires
nltk.download('punkt', quiet=True)
# On charge le modèle français de SpaCy
nlp = spacy.load("fr_core_news_sm")

# Phrase de test complexe en français (avec apostrophes, tirets et ponctuation)
phrase_francaise = "L'étudiant travaille-t-il au laboratoire, aujourd'hui ?"

print("=== COMPARAISON DES TOKENISEURS (FRANÇAIS) ===\n")
print("Texte brut :", phrase_francaise)
print("-" * 50)

# --- 1. Approche par Expression Régulière (Regex) ---
# Ce motif sépare les mots, les lettres avec apostrophes (l', d') et la ponctuation
pattern_regex = r"[A-Za-zÀ-ÿ]+(?:'[A-Za-zÀ-ÿ]+)?|[\.,\?!'\-]"
tokens_regex = re.findall(pattern_regex, phrase_francaise)
print("1. RegEx Tokenizer :")
print(tokens_regex)
print("-" * 50)

# --- 2. Approche par NLTK ---
tokenizer_nltk = WordPunctTokenizer()
tokens_nltk = tokenizer_nltk.tokenize(phrase_francaise)
print("2. NLTK (WordPunct) Tokenizer :")
print(tokens_nltk)
print("-" * 50)

# --- 3. Approche par SpaCy (Industriel) ---
doc_spacy = nlp(phrase_francaise)
tokens_spacy = [token.text for token in doc_spacy]
print("3. SpaCy Tokenizer :")
print(tokens_spacy)
print("=" * 50)

=== COMPARAISON DES TOKENISEURS (FRANÇAIS) ===

Texte brut : L'étudiant travaille-t-il au laboratoire, aujourd'hui ?
--------------------------------------------------
1. RegEx Tokenizer :
["L'étudiant", 'travaille', '-', 't', '-', 'il', 'au', 'laboratoire', ',', "aujourd'hui", '?']
--------------------------------------------------
2. NLTK (WordPunct) Tokenizer :
['L', "'", 'étudiant', 'travaille', '-', 't', '-', 'il', 'au', 'laboratoire', ',', 'aujourd', "'", 'hui', '?']
--------------------------------------------------
3. SpaCy Tokenizer :
["L'", 'étudiant', 'travaille', '-t', '-il', 'au', 'laboratoire', ',', "aujourd'hui", '?']


# DATASET ET DATALOADER

In [85]:
import torch
from torch.utils.data import Dataset, DataLoader

# Notre classe de Dataset personnalisée
class BambaraPOSDataset(Dataset):
    def __init__(self, data, word_to_ix, tag_to_ix): # Initialisation du dataset
        self.data = data # Les données du dataset
        self.word_to_ix = word_to_ix # Dictionnaire de mots (word_to_ix)
        self.tag_to_ix = tag_to_ix # Dictionnaire de tags (tag_to_ix)

    def __len__(self):
        # Renvoie le nombre total de phrases
        return len(self.data)

    def __getitem__(self, idx):
        # Récupère une phrase et ses tags à un index précis
        sentence, tags = self.data[idx]
        
        # On les convertit en tenseurs d'index (sans padding pour l'instant)
        input_tensor = prepare_sequence(sentence, self.word_to_ix)
        target_tensor = prepare_sequence(tags, self.tag_to_ix)
        
        return input_tensor, target_tensor
    
"""
ici , on définit une classe de Dataset personnalisée pour PyTorch, appelée BambaraPOSDataset.
Cette classe hérite de torch.utils.data.Dataset et permet de gérer facilement les données d'entraînement pour le POS tagging en Bambara.
- La méthode __init__ initialise le dataset avec les données, le dictionnaire de mots et le dictionnaire de tags.
- La méthode __len__ renvoie le nombre total de phrases dans le dataset.
- La méthode __getitem__ récupère une phrase et ses tags à un index précis, les convertit en tenseurs d'index et les retourne

"""

"\nici , on définit une classe de Dataset personnalisée pour PyTorch, appelée BambaraPOSDataset.\nCette classe hérite de torch.utils.data.Dataset et permet de gérer facilement les données d'entraînement pour le POS tagging en Bambara.\n- La méthode __init__ initialise le dataset avec les données, le dictionnaire de mots et le dictionnaire de tags.\n- La méthode __len__ renvoie le nombre total de phrases dans le dataset.\n- La méthode __getitem__ récupère une phrase et ses tags à un index précis, les convertit en tenseurs d'index et les retourne\n\n"

In [86]:
from torch.nn.utils.rnn import pad_sequence

# Cette fonction prend un lot (batch) de taille variable et applique le padding
def collate_fn_padd(batch):
    # batch est une liste de tuples (input_tensor, target_tensor)
    inputs = [item[0] for item in batch] # On récupère les tenseurs d'entrée (phrases)
    targets = [item[1] for item in batch] # On récupère les tenseurs de sortie (tags)
    
    # On applique le padding (0 pour les mots, 0 pour les tags)
    padded_inputs = pad_sequence(inputs, batch_first=True, padding_value=0) # On applique le padding sur les entrées
    padded_targets = pad_sequence(targets, batch_first=True, padding_value=0) # On applique le padding sur les sorties
    
    return padded_inputs, padded_targets

"""
en gros du gros , on définit une fonction collate_fn_padd qui est utilisée pour préparer les lots (batches) de données pour l'entraînement.
Cette fonction prend un batch de taille variable (une liste de tuples contenant les tenseurs d'entrée et de sortie) et applique un padding pour uniformiser la longueur des séquences.
Le padding est fait avec l'index 0, qui correspond au token spécial "<PAD>".
La fonction retourne deux tenseurs : padded_inputs (les phrases avec padding) et padded_targets (les tags avec padding ), prêts à être utilisés pour l'entraînement du modèle.

    
"""

'\nen gros du gros , on définit une fonction collate_fn_padd qui est utilisée pour préparer les lots (batches) de données pour l\'entraînement.\nCette fonction prend un batch de taille variable (une liste de tuples contenant les tenseurs d\'entrée et de sortie) et applique un padding pour uniformiser la longueur des séquences.\nLe padding est fait avec l\'index 0, qui correspond au token spécial "<PAD>".\nLa fonction retourne deux tenseurs : padded_inputs (les phrases avec padding) et padded_targets (les tags avec padding ), prêts à être utilisés pour l\'entraînement du modèle.\n\n\n'

In [87]:
# 1. On crée l'instance de notre Dataset
bambara_dataset = BambaraPOSDataset(training_data_multisize, word_to_ix_pad, tag_to_ix_pad)

# 2. On crée le DataLoader
# batch_size=2 signifie qu'il va regrouper nos 3 phrases en paquets de 2 maximum
# shuffle=True mélange les phrases à chaque époque pour un meilleur apprentissage
# collate_fn utilise notre fonction de padding automatique
bambara_loader = DataLoader(
    bambara_dataset, 
    batch_size=2, 
    shuffle=True, 
    collate_fn=collate_fn_padd
)

# 3. Test d'affichage d'un batch
print("--- TEST DU DATALOADER ---")
for batch_idx, (inputs, targets) in enumerate(bambara_loader):
    print(f"\nBatch {batch_idx + 1} :")
    print("Tenseur des entrées (Mots) :\n", inputs)
    print("Shape :", inputs.shape)
    print("Tenseur des cibles (Tags)  :\n", targets)
    print("Shape :", targets.shape)
    

    """
 Ainsi  ici , on crée une instance de notre Dataset personnalisé BambaraPOSDataset avec les données d'entraînement, le dictionnaire de mots et le dictionnaire de tags.
Ensuite, on crée un DataLoader qui va gérer le batching, le mélange des données et l'application du padding automatique grâce à la fonction collate_fn_padd.
On teste ensuite le DataLoader en affichant les tenseurs d'entrée (phrases) et de sortie (tags) pour chaque batch généré. 
On peut voir la forme (shape) de ces tenseurs pour vérifier que tout est correct et que le padding a été appliqué correctement.
    """

--- TEST DU DATALOADER ---

Batch 1 :
Tenseur des entrées (Mots) :
 tensor([[ 2,  3,  4,  5],
        [10, 11,  0,  0]])
Shape : torch.Size([2, 4])
Tenseur des cibles (Tags)  :
 tensor([[1, 2, 3, 1],
        [4, 2, 0, 0]])
Shape : torch.Size([2, 4])

Batch 2 :
Tenseur des entrées (Mots) :
 tensor([[4, 6, 7, 8, 9]])
Shape : torch.Size([1, 5])
Tenseur des cibles (Tags)  :
 tensor([[3, 1, 2, 3, 1]])
Shape : torch.Size([1, 5])


In [88]:
import torch.optim as optim # pour l'optimisation des paramètres du modèle
import torch.nn as nn

# 1. Réinitialisation du modèle et de l'optimiseur
EMBEDDING_DIM = 6
HIDDEN_DIM = 6

model_final = LSTMTaggerWithBatch( # on réinitialise le modèle pour l'entraînement par lots
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    vocab_size=len(word_to_ix_pad),
    tagset_size=len(tag_to_ix_pad)
)

# Très important : ignore_index=0 pour ne pas calculer la perte sur le padding (<PAD>)
loss_function = nn.NLLLoss(ignore_index=0)
optimizer = optim.SGD(model_final.parameters(), lr=0.1)

# 2. Boucle d'entraînement sur 150 époques
epochs = 150
print("Début de l'entraînement par lots...")

for epoch in range(epochs):
    epoch_loss = 0.0
    
    # Le DataLoader nous fournit automatiquement des paquets (batches) d'entrées et de cibles
    for inputs, targets in bambara_loader:
        # Étape a : PyTorch accumule les gradients, on les remet à zéro à chaque lot
        model_final.zero_grad()
        
        # Étape b : Passage des données dans le modèle
        tag_scores = model_final(inputs)
        
        # Étape c : Calcul de la perte. 
        # Attention aux dimensions : PyTorch attend (N, C) ou (N, d1, d2..., C) 
        # On transpose pour aligner les dimensions de scores et de cibles
        loss = loss_function(tag_scores.view(-1, len(tag_to_ix_pad)), targets.view(-1))
        
        # Étape d : Rétropropagation et mise à jour des poids
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    # Affichage de la perte toutes les 25 époques
    if (epoch + 1) % 25 == 0:
        print(f"Époque {epoch + 1:3d}/{epochs} | Perte moyenne : {epoch_loss / len(bambara_loader):.4f}")

print("Entraînement terminé avec succès ! ")

"""
ici , on entraîne le modèle final sur 150 époques en utilisant le DataLoader pour gérer les batches de données.
Pour chaque batch, on remet à zéro les gradients, on passe les données dans le modèle pour obtenir les scores de log-probabilités, on calcule la perte en tenant compte du padding, 
on effectue la rétropropagation pour calculer les gradients, et enfin on met à jour les poids du modèle.
On affiche la perte moyenne toutes les 25 époques pour suivre l'évolution de l'entraînement. À la fin, on indique que l'entraînement est terminé avec succès 


En faisant .view(-1, nombre_de_tags) sur les scores et .view(-1) sur les cibles, on "aplatit" temporairement nos matrices pour que PyTorch puisse calculer la perte de chaque mot un par un, 
tout en ignorant automatiquement l'index 0 (les <PAD>) grâce à ignore_index=0.
    
"""

Début de l'entraînement par lots...
Époque  25/150 | Perte moyenne : 1.3921
Époque  50/150 | Perte moyenne : 1.1934
Époque  75/150 | Perte moyenne : 1.0240
Époque 100/150 | Perte moyenne : 0.7835
Époque 125/150 | Perte moyenne : 0.6360
Époque 150/150 | Perte moyenne : 0.4668
Entraînement terminé avec succès ! 


'\nici , on entraîne le modèle final sur 150 époques en utilisant le DataLoader pour gérer les batches de données.\nPour chaque batch, on remet à zéro les gradients, on passe les données dans le modèle pour obtenir les scores de log-probabilités, on calcule la perte en tenant compte du padding, \non effectue la rétropropagation pour calculer les gradients, et enfin on met à jour les poids du modèle.\nOn affiche la perte moyenne toutes les 25 époques pour suivre l\'évolution de l\'entraînement. À la fin, on indique que l\'entraînement est terminé avec succès \n\n\nEn faisant .view(-1, nombre_de_tags) sur les scores et .view(-1) sur les cibles, on "aplatit" temporairement nos matrices pour que PyTorch puisse calculer la perte de chaque mot un par un, \ntout en ignorant automatiquement l\'index 0 (les <PAD>) grâce à ignore_index=0.\n\n'

In [89]:
# On teste sur la première phrase de notre jeu d'entraînement : "Amadou étudie le bambara"
phrase_test = "Amadou étudie le bambara".split()

# 1. On prépare la séquence avec notre dictionnaire (sans padding car c'est une seule phrase)
inputs_test = prepare_sequence(phrase_test, word_to_ix_pad).unsqueeze(0) # unsqueeze(0) ajoute la dimension de batch (1, 4)

# 2. Mode évaluation et prédiction
model_final.eval()
with torch.no_grad(): # with torch.no_grad() désactive le calcul des gradients pour économiser de la mémoire et accélérer l'inférence
    scores = model_final(inputs_test)
    # On récupère l'index du tag avec la plus grande probabilité pour chaque mot
    predictions = torch.argmax(scores, dim=-1).squeeze(0).tolist()

# 3. Traduction des index en vrais tags
tags_predits = [list(tag_to_ix_pad.keys())[list(tag_to_ix_pad.values()).index(idx)] for idx in predictions]

print("Phrase :", phrase_test)
print("Tags prédits :", tags_predits)


"""en gros , ici , on teste le modèle final sur une phrase connue du jeu d'entraînement.
On prépare la séquence d'entrée en convertissant les mots en indices, puis on passe cette séquence à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag.
"""

Phrase : ['Amadou', 'étudie', 'le', 'bambara']
Tags prédits : ['VERBE', 'VERBE', 'DET', 'NOM']


"en gros , ici , on teste le modèle final sur une phrase connue du jeu d'entraînement.\nOn prépare la séquence d'entrée en convertissant les mots en indices, puis on passe cette séquence à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag.\n"

In [90]:


# Écrivez la phrase que vous voulez tester ici (séparez bien les mots par des espaces)
# Vous pouvez tester des phrases connues comme : "il apprend" ou "le modèle comprend les phrases"
# Ou glisser des mots inconnus comme : "Amadou comprend le bambara"
phrase_a_tester = "Yacouba gère le cool"

# 1. Prétraitement de la phrase
mots_test = phrase_a_tester.split()

# 2. Conversion en index avec gestion des mots inconnus (<UNK>)
index_mots = []
for mot in mots_test:
    if mot in word_to_ix_pad:
        index_mots.append(word_to_ix_pad[mot])
    else:
        index_mots.append(word_to_ix_pad["<UNK>"])  # On remplace par <UNK> si inconnu

# 3. Conversion en tenseur et ajout de la dimension de Batch (1, longueur_phrase)
inputs_test = torch.tensor(index_mots, dtype=torch.long).unsqueeze(0)

# 4. Prédiction du modèle
model_final.eval()
with torch.no_grad():
    scores = model_final(inputs_test)
    predictions = torch.argmax(scores, dim=-1).squeeze(0).tolist()

# 5. Traduction des index prédits en vrais Tags
tags_predits = []
for idx in predictions:
    # On cherche le tag correspondant à l'index (en ignorant le padding)
    tag_nom = list(tag_to_ix_pad.keys())[list(tag_to_ix_pad.values()).index(idx)]
    tags_predits.append(tag_nom)

# 6. Affichage propre du résultat
print("=== RÉSULTAT DU TEST ===")
print(f"Phrase saisie : {mots_test}")
print(f"Tags prédits   : {tags_predits}")
print("========================")

"""
Normalement , ici , on teste le modèle final sur une phrase saisie par l'utilisateur.
On commence par prétraiter la phrase en séparant les mots, puis on convertit chaque mot en son index correspondant dans le dictionnaire, en remplaçant les mots inconnus par le token spécial "<UNK>".
Ensuite, on convertit cette liste d'indices en un tenseur PyTorch et on ajoute une dimension de batch pour que le modèle puisse traiter la phrase.
On passe ce tenseur à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag, puis on récupère les indices des tags prédits en prenant l'argmax.
Enfin, on traduit ces indices en noms de tags lisibles et on affiche le résultat de manière propre pour que l'utilisateur puisse voir la phrase saisie et les tags prédits par le modèle.


Normalement aussi comme là on a des mots inconnus dans la phrase testée, 
le modèle devrait les remplacer par <UNK> et prédire des tags pour ces mots inconnus en se basant sur ce qu'il a appris lors de l'entraînement.  
    """

=== RÉSULTAT DU TEST ===
Phrase saisie : ['Yacouba', 'gère', 'le', 'cool']
Tags prédits   : ['NOM', 'NOM', 'DET', 'NOM']


'\nNormalement , ici , on teste le modèle final sur une phrase saisie par l\'utilisateur.\nOn commence par prétraiter la phrase en séparant les mots, puis on convertit chaque mot en son index correspondant dans le dictionnaire, en remplaçant les mots inconnus par le token spécial "<UNK>".\nEnsuite, on convertit cette liste d\'indices en un tenseur PyTorch et on ajoute une dimension de batch pour que le modèle puisse traiter la phrase.\nOn passe ce tenseur à travers le modèle pour obtenir les scores de log-probabilités pour chaque tag, puis on récupère les indices des tags prédits en prenant l\'argmax.\nEnfin, on traduit ces indices en noms de tags lisibles et on affiche le résultat de manière propre pour que l\'utilisateur puisse voir la phrase saisie et les tags prédits par le modèle.\n\n\nNormalement aussi comme là on a des mots inconnus dans la phrase testée, \nle modèle devrait les remplacer par <UNK> et prédire des tags pour ces mots inconnus en se basant sur ce qu\'il a appris lo

# AVANCEE EN BAMANAKAN

# Le Défi du Bambara : Pourquoi l'alphabet change tout ?

Le français utilise des accents classiques (`é`, `è`, `à`, `ç`). Le bambara, lui, utilise des caractères de l'alphabet de référence national qui cassent les tokeniseurs standards si on ne les configure pas :

* **Les voyelles ouvertes :** `ɛ` (e ouvert) et `ɔ` (o ouvert).
* **Les consonnes nasales / palatales :** `ɲ` (comme dans *ɲuma*) et `ŋ` (comme dans *ŋana*).
* **Les marques de tons ou de nasalisation :** comme le `n` en fin de syllabe.

---

## Le Problème de la Tokenisation Standard

Si on garde le tokeniseur français ou un `.split()` basique, la ponctuation va encore tout fausser. 

De plus, si on utilise une RegEx classique `[a-zA-Z]`, elle va exclure `ɛ`, `ɔ`, `ɲ`, `ŋ` et couper les mots en deux.

### Exemples Concrets de Dysfonctionnement :

* **Exemple 1 (Coupure de mot) :**
  * **Mot d'origine :** `bɛ` (signifie *être* ou *tout*)
  * **Avec RegEx classique `[a-zA-Z]` :** Le caractère `ɛ` est exclu. Le mot est coupé et devient juste `b`.

* **Exemple 2 (Altération de sens) :**
  * **Mot d'origine :** `ɲuma` (signifie *bon*)
  * **Avec RegEx classique `[a-zA-Z]` :** Le `ɲ` est supprimé, le tokeniseur renvoie `uma`, ce qui ne correspond plus au mot initial.

* **Exemple 3 (Rupture de chaîne) :**
  * **Mot d'origine :** `ŋana` (signifie *héros*)
  * **Avec RegEx classique `[a-zA-Z]` :** Le `ŋ` est ignoré, transformant le mot en `ana`.

---

## Solution Technique

Pour éviter cela, on va  configurer le tokeniseur ou la RegEx pour inclure explicitement ces caractères spécifiques :
`[a-zA-ZɛƐɔƆɲƝŋŊ]` (.

## Etape 1 : Mettre à jour le Tokeniseur pour le Bambara

In [91]:
import re # pour les expressions régulières

def bamanankan_tokenizer(text):
    """
    Tokeniseur adapté pour le Bambara (Bamanankan).
    Isorole proprement la ponctuation et préserve les caractères ɛ, ɔ, ɲ, ŋ.
    """
    text = text.strip()
    
    # Motif qui capture les mots contenant les lettres standards + les lettres spécifiques du bambara
    # Et isole la ponctuation classique : . , ! ? ; :
    pattern = r"[a-zA-ZɛɛɔɔɲɲŋŋƐƐƆƆƝƝŊŊ]+|[\.,!\?\(\):;]"
    
    return re.findall(pattern, text)

# --- TEST UNITAIRE ---
phrase_bambara = "Amadou bɛ kalan kɛ, a bɛ bamanankan fɔ ɲuman !"
print("Tokens Bambara :", bamanankan_tokenizer(phrase_bambara))


"""
ici , on définit un tokeniseur spécifique pour le Bambara (Bamanankan) qui utilise des expressions régulières pour capturer les mots et la ponctuation.
Le motif regex est conçu pour reconnaître les lettres standard ainsi que les lettres spécifiques au Bambara (ɛ, ɔ, ɲ, ŋ) et pour isoler la ponctuation classique.

"""

Tokens Bambara : ['Amadou', 'bɛ', 'kalan', 'kɛ', ',', 'a', 'bɛ', 'bamanankan', 'fɔ', 'ɲuman', '!']


'\nici , on définit un tokeniseur spécifique pour le Bambara (Bamanankan) qui utilise des expressions régulières pour capturer les mots et la ponctuation.\nLe motif regex est conçu pour reconnaître les lettres standard ainsi que les lettres spécifiques au Bambara (ɛ, ɔ, ɲ, ŋ) et pour isoler la ponctuation classique.\n\n'

## Étape 2 : Créer le nouveau Jeu de Données d'Entraînement (Bambara)

In [92]:
# Notre nouveau jeu de données en Bambara
# Format : (Phrase brute, Liste des Tags correspondants)
bambara_raw_data = [
    ("Amadou bɛ kalan kɛ .", ["NOM", "VERBE", "NOM", "VERBE", "PUNCT"]),
    ("A bɛ bamanankan fɔ ɲuman .", ["PRON", "VERBE", "NOM", "VERBE", "ADJ", "PUNCT"]),
    ("Modèle bɛ bamanankan kɛ nɔgɔman .", ["NOM", "VERBE", "NOM", "VERBE", "ADJ", "PUNCT"])
]

# Tokenisation automatique de notre dataset
training_data_bambara = []
for sentence, tags in bambara_raw_data:
    tokens = bamanankan_tokenizer(sentence)
    training_data_bambara.append((tokens, tags))

print("Exemple de données tokenisées :")
print(training_data_bambara[0])

Exemple de données tokenisées :
(['Amadou', 'bɛ', 'kalan', 'kɛ', '.'], ['NOM', 'VERBE', 'NOM', 'VERBE', 'PUNCT'])


## Etape 3 : Reconstruire les Dictionnaires Indexés (Vocabulaire

In [93]:
word_to_ix_bambara = {"<PAD>": 0, "<UNK>": 1}
tag_to_ix_bambara = {"<PAD>": 0} # Le padding pour les tags aussi

for words, tags in training_data_bambara:
    for word in words:
        if word not in word_to_ix_bambara:
            word_to_ix_bambara[word] = len(word_to_ix_bambara)
    for tag in tags:
        if tag not in tag_to_ix_bambara:
            tag_to_ix_bambara[tag] = len(tag_to_ix_bambara)

print("Nouveau Vocabulaire Bambara :", word_to_ix_bambara)
print("Nouveaux Tags :", tag_to_ix_bambara)

Nouveau Vocabulaire Bambara : {'<PAD>': 0, '<UNK>': 1, 'Amadou': 2, 'bɛ': 3, 'kalan': 4, 'kɛ': 5, '.': 6, 'A': 7, 'bamanankan': 8, 'fɔ': 9, 'ɲuman': 10, 'Mod': 11, 'le': 12, 'nɔgɔman': 13}
Nouveaux Tags : {'<PAD>': 0, 'NOM': 1, 'VERBE': 2, 'PUNCT': 3, 'PRON': 4, 'ADJ': 5}


## Etape 4 : Instanciation du Dataset et du DataLoader Bambara

In [94]:
# 1. Instanciation du Dataset avec nos données en Bamanankan
bambara_dataset = BambaraPOSDataset(
    training_data_bambara, 
    word_to_ix_bambara, 
    tag_to_ix_bambara
)

# 2. Création du DataLoader avec mélange et lots de 2 phrases
bambara_loader = DataLoader(
    bambara_dataset, 
    batch_size=2, 
    shuffle=True, 
    collate_fn=collate_fn_padd
)

# --- VERIFICATION DES MATRICES (TEST ANALYTIQUE) ---
print("=== VERIFICATION DU BATCHING BAMBARA ===")
for i, (inputs, targets) in enumerate(bambara_loader):
    print(f"\nLot (Batch) N°{i+1} :")
    print(f"  - Matrice des entrées (Mots) : {inputs.shape}")
    print(f"  - Matrice des cibles (Tags)  : {targets.shape}")
    print(f"  - Contenu des entrées encodées :\n    {inputs}")

=== VERIFICATION DU BATCHING BAMBARA ===

Lot (Batch) N°1 :
  - Matrice des entrées (Mots) : torch.Size([2, 7])
  - Matrice des cibles (Tags)  : torch.Size([2, 6])
  - Contenu des entrées encodées :
    tensor([[11, 12,  3,  8,  5, 13,  6],
        [ 7,  3,  8,  9, 10,  6,  0]])

Lot (Batch) N°2 :
  - Matrice des entrées (Mots) : torch.Size([1, 5])
  - Matrice des cibles (Tags)  : torch.Size([1, 5])
  - Contenu des entrées encodées :
    tensor([[2, 3, 4, 5, 6]])


###  Résolution du Bug de Dimensionnement (Pipeline Bambara)

***Quand je suis arrivé ici , il y avait un problème lié à l'input batch_size alors je rédige ce markdown en synthèse et solution de ce problème***

####  Origine du Problème (`ValueError: Expected input batch_size...`)
L'erreur de dimensionnement survenue lors du calcul de la fonction de perte (`NLLLoss`) provenait d'un désalignement entre les données brutes stockées dans le `DataLoader` et l'instanciation des dictionnaires de tokens. 
Deux causes principales ont été identifiées :
1. **Effet mémoire de Jupyter :** Le `DataLoader` tournait en tâche de fond avec les anciennes dimensions du jeu de données précédent (français), tandis que le modèle était réinitialisé avec les nouvelles tailles du vocabulaire bambara.
2. **Incohérence des jetons spéciaux :** L'ajout asymétrique des jetons spéciaux (`<PAD>` et `<UNK>`) a modifié la taille finale du dictionnaire de tags par rapport à celle des mots, créant un décalage mathématique lors de l'aplatissement des matrices avec `.view(-1, NB_TAGS_CLASSES)`.

####  Solutions et Améliorations Apportées
Pour stabiliser l'architecture et basculer proprement sur la langue Bambara, les corrections suivantes ont été intégrées dans un bloc d'exécution unique :

* **Mise à jour du Tokeniseur :** Extension de la RegEx `[a-zA-ZÀ-ÿɛƐɔƆɲƝŋŊ]+` pour supporter simultanément les accents de la langue de travail (français) et les caractères officiels de l'alphabet national du Mali (Bamanankan).
* **Ajout d'un Assert de Sécurité :** Intégration d'une vérification stricte (`assert len(tokens) == len(tags)`) pour garantir, avant tout calcul, que chaque phrase possède exactement le même nombre de mots et d'étiquettes grammaticales.
* **Synchronisation Globale :** Regroupement de la tokenisation, de la reconstruction des index, de l'instanciation du `DataLoader` et de la création du modèle dans la même cellule pour écraser les variables fantômes en mémoire.
* **Calcul Dynamique des Pertes :** La fonction `.view(-1, NB_TAGS_CLASSES)` se base désormais sur la longueur lue en direct du dictionnaire mis à jour, rendant la boucle d'entraînement parfaitement étanche et robuste.

In [ ]:
import torch.nn as nn
import torch.optim as optim
import re # pour les expressions régulières

# 1. CORRECTION DU TOKENISEUR
def bamanankan_tokenizer(text):
    """Tokeniseur adapté pour le Bambara avec support des accents français et caractères bambara."""
    text = text.strip()
    # Inclut toutes les lettres accentuées françaises + caractères bambara
    pattern = r"[a-zA-ZÀ-ÿɛƐɔƆɲƝŋŊ]+|[.,!?'\";:()\[\]]"
    return re.findall(pattern, text)

# 2. RECONSTRUCTION DES DONNÉES AVEC LE BON TOKENISEUR
bambara_raw_data = [
    ("Amadou bɛ kalan kɛ .", ["NOM", "VERBE", "NOM", "VERBE", "PUNCT"]),
    ("A bɛ bamanankan fɔ ɲuman .", ["PRON", "VERBE", "NOM", "VERBE", "ADJ", "PUNCT"]),
    ("Modèle bɛ bamanankan kɛ nɔgɔman .", ["NOM", "VERBE", "NOM", "VERBE", "ADJ", "PUNCT"])
]

# Tokenisation automatique
training_data_bambara = []
for sentence, tags in bambara_raw_data:
    tokens = bamanankan_tokenizer(sentence)
    training_data_bambara.append((tokens, tags))

print("Vérification des données tokenisées :")
for tokens, tags in training_data_bambara:
    print(f"  {tokens} -> {len(tokens)} tokens, {tags} -> {len(tags)} tags")
    assert len(tokens) == len(tags), f" Incohérence : {len(tokens)} tokens vs {len(tags)} tags"

# 3. RECONSTRUCTION DES DICTIONNAIRES
word_to_ix_bambara = {"<PAD>": 0, "<UNK>": 1}
tag_to_ix_bambara = {"<PAD>": 0}

for words, tags in training_data_bambara:
    for word in words:
        if word not in word_to_ix_bambara:
            word_to_ix_bambara[word] = len(word_to_ix_bambara)
    for tag in tags:
        if tag not in tag_to_ix_bambara:
            tag_to_ix_bambara[tag] = len(tag_to_ix_bambara)

# 4. RÉCUPÉRATION DES DIMENSIONS
NB_MOTS_VOCAB = len(word_to_ix_bambara)
NB_TAGS_CLASSES = len(tag_to_ix_bambara)

print("\n=== DIMENSIONS ===")
print(f"Vocabulaire Bambara : {NB_MOTS_VOCAB} mots uniques")
print(f"Tags Classes        : {NB_TAGS_CLASSES} tags uniques")
print(f"word_to_ix_bambara  : {word_to_ix_bambara}")
print(f"tag_to_ix_bambara   : {tag_to_ix_bambara}")

# 5. CRÉATION DU DATASET ET DATALOADER
bambara_dataset = BambaraPOSDataset(
    training_data_bambara, 
    word_to_ix_bambara, 
    tag_to_ix_bambara
)

bambara_loader = DataLoader(
    bambara_dataset, 
    batch_size=2, 
    shuffle=True, 
    collate_fn=collate_fn_padd
)

# 6. VÉRIFICATION DES BATCHES
print("\n=== VÉRIFICATION DES BATCHES ===")
for i, (inputs, targets) in enumerate(bambara_loader):
    print(f"Batch {i+1}: inputs.shape={inputs.shape}, targets.shape={targets.shape}")
    print(f"  inputs  : {inputs}")
    print(f"  targets : {targets}")

# 7. MODÈLE ET ENTRAÎNEMENT
EMBEDDING_DIM = 6
HIDDEN_DIM = 6

model_bambara = LSTMTaggerWithBatch(
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    vocab_size=NB_MOTS_VOCAB,
    tagset_size=NB_TAGS_CLASSES
)

loss_function = nn.NLLLoss(ignore_index=0)
optimizer = optim.SGD(model_bambara.parameters(), lr=0.1)

epochs = 150
print("\nDébut de l'entraînement en Bambara...")

for epoch in range(epochs):
    epoch_loss = 0.0
    for inputs, targets in bambara_loader:
        model_bambara.zero_grad()
        
        tag_scores = model_bambara(inputs)
        
        # 🔥 VÉRIFICATION DES DIMENSIONS AVANT LA LOSS
        # tag_scores: (batch_size, seq_len, nb_tags)
        # targets: (batch_size, seq_len)
        # On les aplatit pour NLLLoss
        loss = loss_function(tag_scores.view(-1, NB_TAGS_CLASSES), targets.view(-1))
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    if (epoch + 1) % 25 == 0:
        print(f"Époque {epoch + 1:3d}/{epochs} | Perte moyenne : {epoch_loss / len(bambara_loader):.4f}")

print("\n✅ Entraînement terminé avec succès 🇲🇱")

Vérification des données tokenisées :
  ['Amadou', 'bɛ', 'kalan', 'kɛ', '.'] -> 5 tokens, ['NOM', 'VERBE', 'NOM', 'VERBE', 'PUNCT'] -> 5 tags
  ['A', 'bɛ', 'bamanankan', 'fɔ', 'ɲuman', '.'] -> 6 tokens, ['PRON', 'VERBE', 'NOM', 'VERBE', 'ADJ', 'PUNCT'] -> 6 tags
  ['Modèle', 'bɛ', 'bamanankan', 'kɛ', 'nɔgɔman', '.'] -> 6 tokens, ['NOM', 'VERBE', 'NOM', 'VERBE', 'ADJ', 'PUNCT'] -> 6 tags

=== DIMENSIONS ===
Vocabulaire Bambara : 13 mots uniques
Tags Classes        : 6 tags uniques
word_to_ix_bambara  : {'<PAD>': 0, '<UNK>': 1, 'Amadou': 2, 'bɛ': 3, 'kalan': 4, 'kɛ': 5, '.': 6, 'A': 7, 'bamanankan': 8, 'fɔ': 9, 'ɲuman': 10, 'Modèle': 11, 'nɔgɔman': 12}
tag_to_ix_bambara   : {'<PAD>': 0, 'NOM': 1, 'VERBE': 2, 'PUNCT': 3, 'PRON': 4, 'ADJ': 5}

=== VÉRIFICATION DES BATCHES ===
Batch 1: inputs.shape=torch.Size([2, 6]), targets.shape=torch.Size([2, 6])
  inputs  : tensor([[ 2,  3,  4,  5,  6,  0],
        [11,  3,  8,  5, 12,  6]])
  targets : tensor([[1, 2, 1, 2, 3, 0],
        [1, 2, 1, 2, 